In [ ]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
from pyathena import connect
from geopy.geocoders import Nominatim
from time import sleep

# Load predictions data from S3

In [ ]:

# Load directly from predictions output
df = pd.read_csv("s3://vegetation-risk-ml/ml/predictions/predictions.csv")

print(df.shape)
df.head()

# Add County Name

In [ ]:
# Mapping of county codes to names based on California county codes
county_map = {
    1: "Alameda", 3: "Alpine", 5: "Amador", 7: "Butte", 9: "Calaveras",
    11: "Colusa", 13: "Contra Costa", 15: "Del Norte", 17: "El Dorado",
    19: "Fresno", 21: "Glenn", 23: "Humboldt", 25: "Imperial",
    27: "Inyo", 29: "Kern", 31: "Kings", 33: "Lake", 35: "Lassen",
    37: "Los Angeles", 39: "Madera", 41: "Marin", 43: "Mariposa",
    45: "Mendocino", 47: "Merced", 49: "Modoc", 51: "Mono",
    53: "Monterey", 55: "Napa", 57: "Nevada", 59: "Orange",
    61: "Placer", 63: "Plumas", 65: "Riverside", 67: "Sacramento",
    69: "San Benito", 71: "San Bernardino", 73: "San Diego",
    75: "San Francisco", 77: "San Joaquin", 79: "San Luis Obispo",
    81: "San Mateo", 83: "Santa Barbara", 85: "Santa Clara",
    87: "Santa Cruz", 89: "Shasta", 91: "Sierra", 93: "Siskiyou",
    95: "Solano", 97: "Sonoma", 99: "Stanislaus",
    101: "Sutter", 103: "Tehama", 105: "Trinity",
    107: "Tulare", 109: "Tuolumne", 111: "Ventura",
    113: "Yolo", 115: "Yuba"
}
#apply mapping to create county_name column
df['county_name'] = df['countycd'].map(county_map)

print("Missing county names:", df['county_name'].isnull().sum())

# Add city using geopy

In [ ]:
geolocator = Nominatim(user_agent="geo_app")

def get_city(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en', timeout=10)
        address = location.raw.get('address', {})
        return address.get('city') or \
               address.get('town') or \
               address.get('village') or \
               address.get('county')
    except:
        return None

# Use ONLY unique locations to minimize API calls and avoid blocking
unique_locations = df[['latitude','longitude']].drop_duplicates().reset_index(drop=True)

cities = []
for i, row in unique_locations.iterrows():
    city = get_city(row['latitude'], row['longitude'])
    cities.append(city)

    sleep(1)  

    if i % 100 == 0:
        print(f"Processed {i} rows")

unique_locations['city'] = cities

# Merge back
df = df.merge(unique_locations, on=['latitude','longitude'], how='left')

# Fill missing
df['city'] = df['city'].fillna('Unknown')

In [ ]:
# Rename for map visuals
df.rename(columns={'lat': 'latitude','lon': 'longitude'}, inplace=True)

# Drop bad rows
df = df.dropna(subset=['latitude', 'longitude', 'risk_score','county_name'])

print("Cleaned:", df.shape)

# Dashboard Features

In [ ]:
# Remove null risk
df = df.dropna(subset=["risk_score"])

# Risk levels
df["risk_level"] = pd.cut(
    df["risk_score"],
    bins=[0, 30, 60, 100],
    labels=["Low Priority", "Medium Priority", "High Priority"],
    include_lowest=True
).astype(str)

# KPI fields
df["trimming_priority_score"] = df["risk_score"].round(1)

# handle null biomass by filling with 0 (assuming no data means no biomass)
df["biomass_tons_ha"] = df["regional_drybiot"].fillna(0)

# High risk flag
df["is_high_risk"] = (df["risk_score"] > 60).astype(int)

print("Feature engineering done")

In [ ]:

# Validate required columns for dashboard

required_cols = [
    "countycd", "county_name", "city", "latitude", "longitude", "risk_score",
    "trimming_priority_score", "risk_level", "biomass_tons_ha", "is_high_risk",
    "fuel_moisture_risk", "fire_recurrence", "log_fire_size",
    "avg_temp", "avg_rain", "avg_wind", "slope"
]
# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
# If any required columns are missing, raise an error
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns present")

# Save to S3

In [ ]:

# Plot-level data to s3 fo
df[required_cols].to_csv("s3://vegetation-risk-ml/dashboard/plots/predictions_clean.csv",index=False)

# Also save a sample locally for quick testing
df[required_cols].sample(2000).to_csv("dashboard/sample_predictions.csv", index=False)


In [ ]:

# County summary (for charts)
county_summary = df.groupby(["county_name", "countycd", "risk_level"]).agg(
    avg_risk_score=("risk_score", "mean"),
    total_plots=("risk_score", "count"),
    high_risk_plots=("is_high_risk", "sum"),
    avg_biomass=("biomass_tons_ha", "mean"),
    avg_moisture=("fuel_moisture_risk", "mean"),
    avg_fire_recurrence=("fire_recurrence", "mean"),
    avg_fire_size=("log_fire_size", "mean"),
    avg_temp=("avg_temp", "mean"),
    avg_rain=("avg_rain", "mean"),
    avg_wind=("avg_wind","mean"),   
    avg_slope=("slope","mean"),  
    avg_lat=("latitude", "mean"),
    avg_lon=("longitude", "mean")
).reset_index()

# Save county summary to S3 for dashboard
county_summary.to_csv("s3://vegetation-risk-ml/dashboard/county/county_summary.csv",index=False)
print("Files saved to S3")

# Connect to Athena

In [ ]:
# Connect to Athena for dashboard queries
s3_staging_dir = "s3://vegetation-risk-ml/athena-results/"

conn = connect(s3_staging_dir=s3_staging_dir,region_name="us-east-1")
cursor = conn.cursor()

print("Connected to Athena")

# Create Table

In [ ]:

# Create external table in Athena for dashboard queries
cursor.execute("DROP TABLE IF EXISTS vegetation_ml.predictions_clean")

create_dbtable = """
CREATE EXTERNAL TABLE vegetation_ml.predictions_clean (
    countycd INT,
    county_name STRING,
    city STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    risk_score DOUBLE,
    trimming_priority_score DOUBLE,
    risk_level STRING,
    biomass_tons_ha DOUBLE,
    is_high_risk INT,
    fuel_moisture_risk DOUBLE,
    fire_recurrence DOUBLE,
    log_fire_size DOUBLE,
    avg_temp DOUBLE,
    avg_rain DOUBLE,
    avg_wind DOUBLE,
    slope DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
LOCATION 's3://vegetation-risk-ml/dashboard/plots/'
TBLPROPERTIES ("skip.header.line.count"="1")
"""

cursor.execute(create_dbtable)

print("Table for dashboard created")

# Create Views in Athena

In [ ]:

# Create dashboard view
create_view = """
CREATE OR REPLACE VIEW vegetation_ml.dashboard_view AS
SELECT
    county_name,
    countycd,
    city,
    risk_level,
    ROUND(AVG(risk_score),1) AS avg_risk_score,
    COUNT(*) AS total_plots,
    SUM(is_high_risk) AS high_risk_plots,
    ROUND(AVG(biomass_tons_ha),1) AS avg_biomass,
    ROUND(AVG(fuel_moisture_risk),1) AS avg_moisture,
    ROUND(AVG(fire_recurrence),1) AS avg_fire_recurrence,
    ROUND(AVG(log_fire_size),1) AS avg_fire_size,
    ROUND(AVG(avg_temp),1) AS avg_temp,
    ROUND(AVG(avg_rain),1) AS avg_rain,
    ROUND(AVG(avg_wind),1) AS avg_wind,
    ROUND(AVG(slope),1) AS avg_slope,
    AVG(latitude) AS avg_lat,
    AVG(longitude) AS avg_lon
FROM vegetation_ml.predictions_clean
GROUP BY county_name, countycd, risk_level
"""

cursor.execute(create_view)

print("Views for dashboard created")

# Validate Query

In [ ]:
# Test query
test = pd.read_sql("SELECT * FROM vegetation_ml.dashboard_view LIMIT 10", conn)
print(test.head())
print("Dashboard prep complete")